# Fraud / Anomaly Detection
## Notebook 05 — Preprocessing and Scaling

### Objective
Compare common scalers, create the canonical robust-scaled feature matrix, and save a fitted preprocessor.

## Imports and data

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.fraud_detection import load_features_labels, evaluate_scores, save_results
X, y = load_features_labels(ROOT)
RANDOM_STATE = 42
print('Features:', X.shape, '| fraud:', int(y.sum()))
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
import joblib

**Interpretation.** Labels are loaded only for later evaluation; they are never passed to a scaler.

**ML decision.** Use all numeric engineered features and fit transformations without target leakage.

## Compare transformations

In [ ]:
scalers = {'standard': StandardScaler(), 'robust': RobustScaler(), 'minmax': MinMaxScaler()}
summary = []
for name, scaler in scalers.items():
    transformed = scaler.fit_transform(X)
    summary.append({'scaler': name, 'min': transformed.min(), 'max': transformed.max(), 'mean_abs': np.abs(transformed).mean()})
pd.DataFrame(summary)

**Interpretation.** Standard scaling is useful for approximately symmetric variables, robust scaling is less affected by extreme transaction values, and min-max scaling has bounded output.

**ML decision.** Choose RobustScaler because genuine fraud-related extremes should not dominate preprocessing.

## Fit and save canonical preprocessing

In [ ]:
scaler = RobustScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
assert np.isfinite(X_scaled.to_numpy()).all()
(ROOT/'data/processed').mkdir(parents=True, exist_ok=True)
(ROOT/'models').mkdir(exist_ok=True)
X_scaled.to_csv(ROOT/'data/processed/creditcard_scaled.csv', index=False)
joblib.dump(scaler, ROOT/'models/robust_scaler.joblib')
X_scaled.describe().T.head()

**Interpretation.** The saved matrix and fitted scaler make all subsequent experiments reproducible.

**ML decision.** Reuse this exact representation for model comparison.